# 사진 속 인원·차량 집계
시작: 2026-10-09 14:45 KST · 종료: 16:35 KST
문제·채점 기준은 README.md를 확인하세요. 아래 기본 예제를 개선하고 submission.csv를 저장하세요.
서버 전체에서 커널 하나만 사용합니다. 완료 후 Kernel → Shutdown으로 실행 자리를 반납하세요.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from ultralytics import YOLO
DATA=Path('data')
sample=pd.read_csv(DATA/'sample_submission.csv')
train_labels=pd.read_csv(DATA/'train_labels.csv')
print(train_labels.shape, sample.shape)


In [ ]:
model=YOLO('yolov8n.pt')
# TODO: 공개 train_images에서 신뢰도·IoU를 검증하세요. COCO class ID: person=0, car=2, bus=5, truck=7
rows=[]
for name in sample.filename:
    result=model.predict(str(DATA/'test_images'/name),device='cpu',imgsz=416,conf=0.25,workers=0,verbose=False)[0]
    classes=result.boxes.cls.cpu().numpy().astype(int)
    row={'filename':name,'person_count':int((classes==0).sum())}
    row['vehicle_count']=int(np.isin(classes,[2,5,7]).sum())
    rows.append(row)
submission=pd.DataFrame(rows)[sample.columns]
submission.to_csv('submission.csv',index=False)
print(submission.head())


In [ ]:
check = pd.read_csv('submission.csv')
assert check.columns.tolist() == sample.columns.tolist()
assert check.iloc[:,0].tolist() == sample.iloc[:,0].tolist()
assert check.iloc[:,0].is_unique and not check.isna().any().any()
print('제출 형식 확인:', check.shape)
